<a href="https://colab.research.google.com/github/elvissoares/EQE595-SimMol/blob/main/notebooks/9_Proteina_Analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Para usar o OpenMM no Google Colab devemos fazer o seguinte passo:
1. Instalar o `openmm[cuda12]`, `mdtraj` e `pdbfixer` via `pip`

In [ ]:
!pip install openmm[cuda12] mdtraj pdbfixer

2. Testando se a instalação deu certo e quais `Platform` estão disponíveis

In [ ]:
!python -m openmm.testInstallation

# Aula Prática 09 - Proteína em Água

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout

# 0. Abrindo Google Drive para salvar arquivos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# path to output files
path = '/content/drive/MyDrive/SimMol/'
# path = '../results/'

# 1. Parâmetros principais

In [ ]:
pdb_id = '1AKI'

temperature = 300.0 * kelvin
pressure = 1.0 * atmosphere
ph = 7.0

ionic_strength = 0.15 * molar

timestep = 2.0 * femtoseconds

output_dir = path + f"md_{pdb_id}"
os.makedirs(output_dir, exist_ok=True)

# 2. Carregando trajetória com MDTraj

In [ ]:
import mdtraj as md

traj = md.load(
    f"{output_dir}/{pdb_id}_production.dcd",
    top=f"{output_dir}/{pdb_id}_after_npt.pdb",
)

protein_atoms = traj.topology.select("protein")
ca_atoms = traj.topology.select("name CA")
backbone_atoms = traj.topology.select("backbone")

traj.superpose(traj, 0, atom_indices=backbone_atoms)

# 3. Calculando RMSD

RMSD = Root Mean Square Deviation

$\text{RMSD} = \sqrt{\frac{1}{n}\sum_{i=1}^n (\boldsymbol{R}_i-\boldsymbol{R}_i^\text{ref})^2}$

RMSD quantifica o quanto a estrutura diverge a partir de um tempo de referência

In [ ]:
rmsd_ca = md.rmsd(traj, traj, 0, atom_indices=ca_atoms)
rmsd_backbone = md.rmsd(traj, traj, 0, atom_indices=backbone_atoms)
rmsd_protein = md.rmsd(traj, traj, 0, atom_indices=protein_atoms)

time_ns = traj.time * 2e-3 # convert from frames to ns

plt.plot(time_ns, rmsd_ca, label="C-alpha")
plt.plot(time_ns, rmsd_backbone, label="Backbone")
plt.plot(time_ns, rmsd_protein, label="Protein")
plt.xlabel("Tempo (ns)")
plt.ylabel("RMSD (nm)")
plt.legend()
plt.tight_layout()
plt.show()

print("RMSD C-alpha médio / nm:", np.mean(rmsd_ca))
print("RMSD backbone médio / nm:", np.mean(rmsd_backbone))
print("RMSD proteína médio / nm:", np.mean(rmsd_protein))

In [ ]:
plt.hist(rmsd_ca, bins=50, density=True)

# construindo uma gaussiana com média e desvio padrão do RMSD C-alpha
rarray = np.mean(rmsd_ca) + np.arange(-5,5,0.01)*np.std(rmsd_ca)
plt.plot(rarray,np.sqrt(1/(2*np.pi*np.std(rmsd_ca)**2))*np.exp(-0.5*(rarray-np.mean(rmsd_ca))**2/np.std(rmsd_ca)**2),color='k')


plt.xlabel("RMSD $C_\\alpha$ (nm)")
plt.ylabel("Densidade de probabilidade")

# 4. Calculando RMSF

RMSF = Root Mean Square Fluctuation

$RMSF = \sqrt{\frac{1}{n}\sum_{i=1}^n(\boldsymbol{R}_i-\langle\boldsymbol{R}_i\rangle)^2}$

onde $\langle\boldsymbol{R}_i\rangle$ é a posição média no ensemble. 


RMSF quantifica quais regiões da proteína são mais flexíveis.

In [ ]:
traj.superpose(traj, 0, atom_indices=backbone_atoms)

rmsf_ca = md.rmsf(traj, traj, frame=0, atom_indices=ca_atoms)

ca_residues = [traj.topology.atom(i).residue for i in ca_atoms]
residue_ids = [res.index for res in ca_residues]
residue_names = [str(res) for res in ca_residues]

plt.plot(residue_ids, rmsf_ca, marker="o")

top3 = np.argsort(rmsf_ca)[-3:][::-1] # acha os 3 resíduos mais flexíveis

for idx in top3:
    plt.annotate(
        residue_names[idx],
        xy=(residue_ids[idx], rmsf_ca[idx]),
        xytext=(residue_ids[idx] + 1, rmsf_ca[idx] + 0.02),
        arrowprops=dict(arrowstyle="->", color="red"),
        fontsize=10,
        color="red",
    )

plt.xlabel("Índice do resíduo")
plt.ylabel("RMSF C-alpha / nm")
plt.tight_layout()
plt.show()

print("Resíduos mais flexíveis:")
for idx in top3:
    print(idx, residue_names[idx], rmsf_ca[idx], "nm")

# 5. Calculando Rg

O raio de giro é calculado a partir das posições dos átomos em relação ao centro de massa da proteína:
$R_g = \sqrt{\frac{\sum_i m_i (\boldsymbol{R}_i-\boldsymbol{R}_i^\text{CM})^2}{\sum_i m_i}}$

O raio de giro ($R_g$) é uma medida da compactação da proteína durante a simulação molecular.
- Se o $R_g$ aumenta, a proteína pode estar se expandindo ou começando a desenovelar;
- Se o $R_g$ diminui, pode estar ficando mais compacta;
- Se o $R_g$ permanece aproximadamente constante ao longo do tempo, isso sugere que a proteína manteve sua compactação estrutural.

In [ ]:
protein_traj = traj.atom_slice(protein_atoms)

rg = md.compute_rg(protein_traj)

plt.plot(time_ns, rg)
plt.xlabel("Tempo (ns)")
plt.ylabel("Raio de giro (nm)")
plt.tight_layout()
plt.show()

print("Rg médio / nm:", np.mean(rg))
print("Rg desvio padrão / nm:", np.std(rg))

In [ ]:
plt.hist(rg, bins=50, density=True)

# construindo uma gaussiana com média e desvio padrão do Rg
rarray = np.mean(rg) + np.arange(-5,5,0.01)*np.std(rg)
plt.plot(rarray,np.sqrt(1/(2*np.pi*np.std(rg)**2))*np.exp(-0.5*(rarray-np.mean(rg))**2/np.std(rg)**2),color='k')


plt.xlabel("Raio de giro (nm)")
plt.ylabel("Densidade de probabilidade")

# BÔNUS. Ramachandran analysis


A cadeia principal de uma proteína é formada por uma repetição:

N — Cα — C — N — Cα — C

Os dois principais ângulos diedrais do backbone são:
$\phi$ - ligação N — Cα	- rotação antes do Cα
$\psi$ - ligação Cα — C	- rotação depois do Cα

Esses ângulos determinam a conformação local da cadeia principal da proteína.

O Ramachandran plot mostra se a geometria local da proteína está em regiões conformacionais fisicamente permitidas e compatíveis com hélices, folhas beta ou regiões desordenadas

In [ ]:
# Calcular ângulos phi e psi

phi_indices, phi = md.compute_phi(protein_traj)
psi_indices, psi = md.compute_psi(protein_traj)

# phi e psi estão em radianos
phi_deg = np.degrees(phi)
psi_deg = np.degrees(psi)

# Ramachandran plot

plt.figure(figsize=(6, 6))

plt.scatter(
    phi_deg.flatten(),
    psi_deg.flatten(),
    s=2,
    alpha=0.1,
)

plt.xlim(-180, 180)
plt.ylim(-180, 180)

plt.xlabel(r"$\phi$ (°)")
plt.ylabel(r"$\psi$ (°)")
plt.title(f"Ramachandran plot - {pdb_id}")

plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)

plt.tight_layout()
# plt.savefig(f"ramachandran_{pdb_id}.png", dpi=300)
plt.show()

Por resíduo 

In [ ]:
residue_to_plot = 0

# Atenção: índices em Python começam em 0.
# Aqui estamos usando a coluna correspondente ao índice interno do array.
phi_res = phi_deg[:, residue_to_plot]
psi_res = psi_deg[:, residue_to_plot]

plt.figure(figsize=(6, 6))

plt.scatter(
    phi_res,
    psi_res,
    s=8, color='C1',
    alpha=0.5,
)

plt.xlim(-180, 180)
plt.ylim(-180, 180)

plt.xlabel(r"$\phi$ (°)")
plt.ylabel(r"$\psi$ (°)")
plt.title(f"Ramachandran - resíduo {residue_names[residue_to_plot]}")

plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)

plt.tight_layout()
plt.show()